# Environment

## Imports

In [17]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
# Reload only modules imported with %aimport
%autoreload 1

# Mark these modules for auto-reload
%aimport fetch_series.core

In [19]:
import os
import httpx
from httpx import RemoteProtocolError, ConnectError, ReadError
from httpx_retries import Retry, RetryTransport
import pandas as pd
import asyncio
import json
import re
import time
from tqdm.auto import tqdm
import fetch_series
from fetch_series.core import *
from typing import List, Tuple, Dict, Any

## Global variables

In [20]:
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

In [21]:
retry = Retry(
    total=5,
    backoff_factor=0.3,  # Wait 0.3s, 0.6s, 1.2s, 2.4s, 4.8s between retries
    status_forcelist=[
        408,  # Request Timeout - server took too long to respond
        429,  # Too Many Requests - rate limiting (common with NCBI)
        500,  # Internal Server Error - temporary server issue
        502,  # Bad Gateway - gateway/proxy error
        503,  # Service Unavailable - server overloaded/maintenance
        504,  # Gateway Timeout - gateway didn't get response in time
    ],
)

transport = RetryTransport(retry=retry)

## Helper functions

In [22]:
def eutils_link(
    dbfrom: str,
    db: str,
    ids: str | None = None,
    webenv: str | None = None,
    query_key: str | None = None,
    retmode: str = "json",
    cmd: str = "neighbor_history",
) -> Dict[str, Any]:
    """
    Search for linked items in a target database using the E-utilities API.
    Args:
        dbfrom (str): database to search from
        db (str): database to search in
        id (str): ID of the item to link
        retmode (str): return mode for the response

    Returns:
        Dict[str, Any]: JSON response from the E-utilities API containing search results
    """
    # Check that either ids or webenv and query_key are specified
    if (ids is None) == (webenv is None or query_key is None):
        raise ValueError("Must specify either ids OR webenv and query_key")

    # Retrieve linked items using the E-utilities API
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"
    params = {
        "dbfrom": dbfrom,
        "db": db,
        "id": ids,
        "retmode": retmode,
        "WebEnv": webenv,
        "query_key": query_key,
        "cmd": cmd,
    }
    with httpx.Client(transport=transport) as client:
        response = client.get(base, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

# Fetching BioProject and BioSample Metadata

## Take a look at the BioProject and BioSample databses with e-tools api

### BioProject

In [23]:
base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi"
params = {
    "db": "bioproject",
    "retmode": "json",
}

response = httpx.get(base, params=params)
response.raise_for_status()
data = response.json()
fields = data["einforesult"]["dbinfo"][0]["fieldlist"].copy()
links = data["einforesult"]["dbinfo"][0]["linklist"].copy()
del data["einforesult"]["dbinfo"][0]["fieldlist"]
del data["einforesult"]["dbinfo"][0]["linklist"]
print(json.dumps(data, indent=2))

{
  "header": {
    "type": "einfo",
    "version": "0.3"
  },
  "einforesult": {
    "dbinfo": [
      {
        "dbname": "bioproject",
        "menuname": "BioProject",
        "description": "BioProject Database",
        "dbbuild": "Build260512-1450.1",
        "count": "1046296",
        "lastupdate": "2026/05/12 15:30"
      }
    ]
  }
}


In [24]:
pd.DataFrame(fields)

,name,fullname,description,termcount,isdate,isnumerical,singletoken,hierarchy,ishidden
0,ALL,All Fields,All terms from all searchable fields,23058615,N,N,N,N,N
1,UID,UID,Unique number assigned to publication,0,N,Y,Y,N,Y
2,FILT,Filter,Limits the records,113,N,N,Y,N,N
3,ORGN,Organism,Organism,1637965,N,N,Y,Y,N
4,PRJA,Project Accession,Project Accession,1327731,N,N,Y,N,N
5,TYPE,Project Type,Project Type,2,N,N,Y,N,N
6,STPE,Project Subtype,Project Subtype,7,N,N,Y,N,N
7,DATE,Registration Date,Registration Date,8531,Y,N,Y,N,N
8,TITL,Title,Title,2587080,N,N,Y,N,N
9,CEN,Submitter Organization,Submitter Organization(s),178305,N,N,Y,N,N


In [25]:
pd.DataFrame(links)

,name,menu,description,dbto
0,bioproject_assembly_all,Assembly Links,All related Assemblies,assembly
1,bioproject_bioproject,BioProject,Links from project to related projects,bioproject
2,bioproject_bioproject_d2u,Umbrella projects,All Umbrella projects,bioproject
3,bioproject_bioproject_u2d,Data projects,All Data projects,bioproject
4,bioproject_biosample_all,BioSample Links,All related BioSamples,biosample
5,bioproject_dbvar,dbVar,Link from BioProjects to dbVar,dbvar
6,bioproject_gap,dbGaP Links,dbGaP Links,gap
7,bioproject_gds,GEO DataSet Links,GEO DataSet links,gds
8,bioproject_genome,Genome Links,Related Genomes,genome
9,bioproject_nuccore,Nucleotide Links,Related Nucleotide entry,nuccore


### BioProject

In [26]:
base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi"
params = {
    "db": "biosample",
    "retmode": "json",
}

response = httpx.get(base, params=params)
response.raise_for_status()
data = response.json()
fields = data["einforesult"]["dbinfo"][0]["fieldlist"].copy()
links = data["einforesult"]["dbinfo"][0]["linklist"].copy()
del data["einforesult"]["dbinfo"][0]["fieldlist"]
del data["einforesult"]["dbinfo"][0]["linklist"]
print(json.dumps(data, indent=2))

{
  "header": {
    "type": "einfo",
    "version": "0.3"
  },
  "einforesult": {
    "dbinfo": [
      {
        "dbname": "biosample",
        "menuname": "BioSample",
        "description": "BioSample Database",
        "dbbuild": "Build260512-2031m.1",
        "count": "56261590",
        "lastupdate": "2026/05/13 00:29"
      }
    ]
  }
}


In [27]:
pd.DataFrame(fields)

,name,fullname,description,termcount,isdate,isnumerical,singletoken,hierarchy,ishidden
0,ALL,All Fields,All terms from all searchable fields,485005332,N,N,N,N,N
1,UID,UID,Unique number assigned to publication,0,N,Y,Y,N,Y
2,FILT,Filter,Limits the records,296,N,N,Y,N,N
3,ACCN,Accession,Accession number of sequence,103163452,N,N,Y,N,N
4,TITL,Title,Words in definition line,19796426,N,N,N,N,N
5,PROP,Properties,Classification by source qualifiers and molecu...,4221,N,N,Y,N,N
6,WORD,Text Word,Free text associated with record,326505677,N,N,N,N,N
7,ORGN,Organism,"Scientific and common names of organism, and a...",1749493,N,N,Y,Y,N
8,AUTH,Author,Author(s) of publication,193436,N,N,Y,N,N
9,PDAT,Publication Date,Date sequence added to GenBank,9834,Y,N,Y,N,N


In [28]:
pd.DataFrame(links)

,name,menu,description,dbto
0,biosample_assembly,Assembly links,Assembly,assembly
1,biosample_biocollections,BioCollections,BioCollections,biocollections
2,biosample_bioproject,BioProject Links,BioProject links,bioproject
3,biosample_dbvar,dbVar Links,Links to dbVar,dbvar
4,biosample_gap,dbGaP Links,Links to dbGap Studies,gap
5,biosample_gds,GEO DataSets Links,GEO DataSets links,gds
6,biosample_nuccore,Nucleotide Links,Nucleotide links,nuccore
7,biosample_omim,OMIM links,OMIM links,omim
8,biosample_pubmed,PubMed Links,PubMed links,pubmed
9,biosample_snp,SNP Links,Related SNP record,snp


## Let's try searching for a specific BioProject

In [29]:
db = "bioproject"
search_results = eutils_search(query="PRJNA988806[PRJA]", db=db, api_key=NCBI_API_KEY)
print(json.dumps(search_results, indent=2))

{
  "header": {
    "type": "esearch",
    "version": "0.3"
  },
  "esearchresult": {
    "count": "1",
    "retmax": "1",
    "retstart": "0",
    "querykey": "1",
    "webenv": "MCID_6a0459c080945971580166e3",
    "idlist": [
      "988806"
    ],
    "translationset": [],
    "translationstack": [
      {
        "term": "PRJNA988806[PRJA]",
        "field": "PRJA",
        "count": "1",
        "explode": "N"
      },
      "GROUP"
    ],
    "querytranslation": "PRJNA988806[PRJA]"
  }
}


In [31]:
summary = eutils_summary(
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
    db=db,
    api_key=NCBI_API_KEY,
)
print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "988806"
    ],
    "988806": {
      "uid": "988806",
      "taxid": 10090,
      "project_id": 988806,
      "project_acc": "PRJNA988806",
      "project_type": "Primary submission",
      "project_data_type": "Transcriptome or Gene expression",
      "sort_by_projecttype": 324537,
      "sort_by_datatype": 300824,
      "sort_by_organism": 421684,
      "project_subtype": "",
      "project_target_scope": "Multiisolate",
      "project_target_material": "Transcriptome",
      "project_target_capture": "Whole",
      "project_methodtype": "Sequencing",
      "project_method": "",
      "project_objectives_list": [
        {
          "project_objectivestype": "Expression",
          "project_objectives": ""
        }
      ],
      "registration_date": "2023/06/28 00:00",
      "project_name": "Aspartate signaling increases the aggressiveness of lung metastases by inducing eIF5A-mediat

In [32]:
links = eutils_link(
    dbfrom="bioproject",
    db="gds",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
)
print(json.dumps(links, indent=2))

{
  "header": {
    "type": "elink",
    "version": "0.3"
  },
  "linksets": [
    {
      "dbfrom": "bioproject",
      "ids": [
        "988806"
      ],
      "linksetdbhistories": [
        {
          "dbto": "gds",
          "linkname": "bioproject_gds",
          "querykey": "2"
        }
      ],
      "webenv": "MCID_6a0459c080945971580166e3"
    }
  ]
}


In [34]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][0]["querykey"],
    db="gds",
    api_key=NCBI_API_KEY,
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "200236084"
    ],
    "200236084": {
      "uid": "200236084",
      "accession": "GSE236084",
      "gds": "",
      "title": "Aspartate signaling increases the aggressiveness of lung metastases by inducing eIF5A-mediated translation (scRNA-Seq)",
      "summary": "Lung metastases are detected in more than half of patients with metastatic tumors. However, it remains largely unknown why the lung environment is a permissive niche for metastases. Here, we discover that pulmonary aspartate triggers a cellular signaling cascade in disseminated cancer cells resulting in a translational program that boosts lung metastasis. Specifically, we observe that patients and mice with breast cancer have high concentrations of aspartate in their lung interstitial fluid. This extracellular aspartate activates the ionotropic N-methyl-D-aspartate (NMDA) receptor in cancer cells, which induces CREB-dependen

In [35]:
links = eutils_link(
    dbfrom="bioproject",
    db="sra",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
    api_key=NCBI_API_KEY,
)
print(json.dumps(links, indent=2))

TypeError: eutils_link() got an unexpected keyword argument 'api_key'

In [ ]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][0]["querykey"],
    db="gds",
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "28241758",
      "28241757",
      "28241756",
      "28241755",
      "28241754"
    ],
    "28241758": {
      "uid": "28241758",
      "expxml": "  <Summary><Title>GSM7518069: TSF2, Lungs from TSF Injection, Metastatic Colonization (d16); Mus musculus; RNA-Seq</Title><Platform instrument_model=\"Illumina NovaSeq 6000\">ILLUMINA</Platform><Statistics total_runs=\"1\" total_spots=\"332639939\" total_bases=\"39584152741\" total_size=\"13332204657\" load_done=\"true\" cluster_name=\"public\"/></Summary><Submitter acc=\"SRA1663813\" center_name=\"Laboratory of Cellular Metabolism and Metabolic Re\" contact_name=\"GEO Group\" lab_name=\"\"/><Experiment acc=\"SRX20810436\" ver=\"2\" status=\"public\" name=\"GSM7518069: TSF2, Lungs from TSF Injection, Metastatic Colonization (d16); Mus musculus; RNA-Seq\"/><Study acc=\"SRP446371\" name=\"Aspartate signaling increases the aggressiveness of lu

In [36]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][1]["querykey"],
    db="gds",
    api_key=NCBI_API_KEY,
)

print(json.dumps(summary, indent=2))

IndexError: list index out of range

In [ ]:
links = eutils_link(
    dbfrom="bioproject",
    db="biosample",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
)
print(json.dumps(links, indent=2))

{
  "header": {
    "type": "elink",
    "version": "0.3"
  },
  "linksets": [
    {
      "dbfrom": "bioproject",
      "ids": [
        "988806"
      ],
      "linksetdbhistories": [
        {
          "dbto": "biosample",
          "linkname": "bioproject_biosample",
          "querykey": "5"
        },
        {
          "dbto": "biosample",
          "linkname": "bioproject_biosample_all",
          "querykey": "6"
        }
      ],
      "webenv": "MCID_69f490170ecb67ce1905873c"
    }
  ]
}


In [37]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][0]["querykey"],
    db="gds",
    api_key=NCBI_API_KEY,
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "200236084"
    ],
    "200236084": {
      "uid": "200236084",
      "accession": "GSE236084",
      "gds": "",
      "title": "Aspartate signaling increases the aggressiveness of lung metastases by inducing eIF5A-mediated translation (scRNA-Seq)",
      "summary": "Lung metastases are detected in more than half of patients with metastatic tumors. However, it remains largely unknown why the lung environment is a permissive niche for metastases. Here, we discover that pulmonary aspartate triggers a cellular signaling cascade in disseminated cancer cells resulting in a translational program that boosts lung metastasis. Specifically, we observe that patients and mice with breast cancer have high concentrations of aspartate in their lung interstitial fluid. This extracellular aspartate activates the ionotropic N-methyl-D-aspartate (NMDA) receptor in cancer cells, which induces CREB-dependen

In [39]:
links = eutils_link(
    dbfrom="bioproject",
    db="gap",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
)
print(json.dumps(links, indent=2))

{
  "header": {
    "type": "elink",
    "version": "0.3"
  },
  "linksets": [
    {
      "dbfrom": "bioproject",
      "ids": [
        "988806"
      ],
      "webenv": "MCID_6a0459c080945971580166e3"
    }
  ]
}


In [40]:
fields = [
    "study_accession", 
    "secondary_study_accession", 
    "sample_accession", 
    "sample_alias", 
    "secondary_sample_accession", 
    "experiment_accession",
    "experiment_alias", 
    "run_accession", 
    "run_alias"
]

runs = read_enaruns(
    series="PRJNA988806",
    format="json",
    fields=",".join(fields),
)
print(json.dumps(runs, indent=2))

[
  {
    "run_accession": "SRR25056226",
    "study_accession": "PRJNA988806",
    "secondary_study_accession": "SRP446371",
    "sample_accession": "SAMN36028298",
    "sample_alias": "GSM7518068",
    "secondary_sample_accession": "SRS18093900",
    "experiment_accession": "SRX20810435",
    "experiment_alias": "GSM7518068_r1",
    "run_alias": "GSM7518068_r1"
  },
  {
    "run_accession": "SRR25056229",
    "study_accession": "PRJNA988806",
    "secondary_study_accession": "SRP446371",
    "sample_accession": "SAMN36028301",
    "sample_alias": "GSM7518065",
    "secondary_sample_accession": "SRS18093896",
    "experiment_accession": "SRX20810432",
    "experiment_alias": "GSM7518065_r1",
    "run_alias": "GSM7518065_r1"
  },
  {
    "run_accession": "SRR25056225",
    "study_accession": "PRJNA988806",
    "secondary_study_accession": "SRP446371",
    "sample_accession": "SAMN36028297",
    "sample_alias": "GSM7518069",
    "secondary_sample_accession": "SRS18093899",
    "experime

# Let's try retrieving metadata for a list of BioProjects

First, let's load a list of selected BioProjects from a file. This file contains metadata for 10x Genomics single cell RNA-seq datasets, including the BioProject accession, the corresponding GEO dataset accession, the SRA experiment accession, and the species of the samples.

In [41]:
columns = ["sample", "gse", "prj", "srs", "srx", "srr", "species"]
samples10x = pd.read_csv("data/All_10x.sample_table.tsv", sep="\t", names=columns)
samples10x.replace("-", pd.NA, inplace=True)
samples10x["n_srx"] = samples10x["srx"].str.split(",").apply(len)
samples10x["n_srr"] = samples10x["srr"].str.split(",").apply(len)
samples10x.head()

,sample,gse,prj,srs,srx,srr,species,n_srx,n_srr
0,DRS173513,NaN,PRJDB9974,DRS173513,DRX222010,DRR231755,Homo sapiens,1,1
1,DRS173514,NaN,PRJDB9974,DRS173514,DRX222011,DRR231756,Homo sapiens,1,1
2,DRS188691,NaN,PRJDB11756,DRS188691,"DRX288815,DRX288817","DRR299376,DRR299378",Homo sapiens,2,2
3,DRS188692,NaN,PRJDB11756,DRS188692,"DRX288803,DRX288805","DRR299364,DRR299366",Homo sapiens,2,2
4,DRS195626,NaN,PRJDB8838,DRS195626,DRX185722,DRR195270,Mus musculus,1,1


In [42]:
samples10x.describe()

,n_srx,n_srr
count,104442.000000,104442.000000
mean,1.098361,2.515176
std,2.098654,3.992520
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,1.000000,3.000000
max,500.000000,500.000000


Group samples by BioProject

In [43]:
grouped_samples10x = (
    samples10x
    .groupby("prj")
    .agg(
        {
            "sample": pd.Series.nunique,
            "gse": pd.Series.nunique,
            "n_srx": "sum",
            "n_srr": "sum"
        }
    )
    .reset_index()
)
grouped_samples10x.head()

,prj,sample,gse,n_srx,n_srr
0,PRJDB10650,3,0,3,3
1,PRJDB10917,1,0,1,1
2,PRJDB10983,6,0,6,6
3,PRJDB11390,4,0,4,4
4,PRJDB11756,2,0,4,4


Group by project and sample

In [44]:
grouped_samples10x_prj_sample = (
    samples10x.groupby(["prj", "srs"])
    .agg(
        {
            "gse": pd.Series.nunique,
            "n_srx": "sum",
            "n_srr": "sum",
        }
    )
    .reset_index()
)
grouped_samples10x_prj_sample.head()

,prj,srs,gse,n_srx,n_srr
0,PRJDB10650,DRS258235,0,1,1
1,PRJDB10650,DRS258236,0,1,1
2,PRJDB10650,DRS258237,0,1,1
3,PRJDB10917,DRS210942,0,1,1
4,PRJDB10983,DRS256234,0,1,1


## Direct fetching using BioProject accession

### BioProject

Let's take a look at the BioProject query results. This file contains metadata for BioProjects that were retrieved using the BioProject accession numbers from the 10x Genomics datasets

In [45]:
bioproject_direct = pd.read_csv("data/query/results/bioproject_summary.csv")
bioproject_direct.head()

,bioproject_accession,uid,project_type,project_data_type,project_target_material,success
0,PRJDB9974,662260.0,Primary submission,Transcriptome or Gene expression,Transcriptome,True
1,PRJDB11756,735962.0,Primary submission,Transcriptome or Gene expression,Transcriptome,True
2,PRJDB8838,746802.0,Primary submission,Transcriptome or Gene expression,Transcriptome,True
3,PRJDB8796,749960.0,Primary submission,Transcriptome or Gene expression,Transcriptome,True
4,PRJDB11390,783551.0,Primary submission,Transcriptome or Gene expression,Transcriptome,True


Let's take a look at the BioProject projects we were not able to retrieve

In [46]:
bioproject_direct[bioproject_direct.success == False]

,bioproject_accession,uid,project_type,project_data_type,project_target_material,success
40,-,NaN,NaN,NaN,NaN,False
4838,PRJNA857927,NaN,NaN,NaN,NaN,False
9533,PRJNA1136968,NaN,NaN,NaN,NaN,False
10337,PRJNA1190403,NaN,NaN,NaN,NaN,False


At the moment (12.06.2026) I indeed can't see those projects in the BioProject database, so they might have been removed or are not publicly available anymore:

- [NCBI search results](https://www.ncbi.nlm.nih.gov/bioproject/?term=PRJNA1136968%5BPRJA%5D+OR+PRJNA1190403%5BPRJA%5D+OR+PRJNA857927%5BPRJA%5D)



In [47]:
db = "bioproject"
search_results = eutils_search(
    query="PRJNA1136968[PRJA] OR PRJNA1190403[PRJA] OR PRJNA857927[PRJA]",
    db=db,
    api_key=NCBI_API_KEY,
)
print(json.dumps(search_results, indent=2))

{
  "header": {
    "type": "esearch",
    "version": "0.3"
  },
  "esearchresult": {
    "count": "0",
    "retmax": "0",
    "retstart": "0",
    "querykey": "1",
    "webenv": "MCID_6a045a0238ccb66d2904684d",
    "idlist": [],
    "translationset": [],
    "querytranslation": "PRJNA1136968[PRJA] OR PRJNA1190403[PRJA] OR PRJNA857927[PRJA]",
    "errorlist": {
      "phrasesnotfound": [
        "PRJNA1136968[PRJA]",
        "PRJNA1190403[PRJA]",
        "PRJNA857927[PRJA]"
      ],
      "fieldsnotfound": []
    },
    "warninglist": {
      "phrasesignored": [],
      "quotedphrasesnotfound": [],
      "outputmessages": [
        "No items found."
      ]
    }
  }
}


### GEO

### SRA

Load results for SRA direct fetching using BioProject accession numbers

In [48]:
sra_direct = pd.read_csv("data/query/results/bioproject2sra_direct_summary.csv")
sra_direct.head()

,bioproject_accession,success,Run,Experiment,Sample,BioSample,Submission,LibraryStrategy,LibrarySource,LibraryLayout,SampleName,ScientificName
0,PRJDB9974,True,DRR231743,DRX221998,DRS151759,SAMD00229074,DRA010287,RNA-Seq,TRANSCRIPTOMIC,PAIRED,SAMD00229074,Homo sapiens
1,PRJDB9974,True,DRR231744,DRX221999,DRS151760,SAMD00229075,DRA010287,RNA-Seq,TRANSCRIPTOMIC,PAIRED,SAMD00229075,Homo sapiens
2,PRJDB9974,True,DRR231745,DRX222000,DRS151761,SAMD00229076,DRA010287,RNA-Seq,TRANSCRIPTOMIC,PAIRED,SAMD00229076,Homo sapiens
3,PRJDB9974,True,DRR231746,DRX222001,DRS151762,SAMD00229077,DRA010287,RNA-Seq,TRANSCRIPTOMIC,PAIRED,SAMD00229077,Homo sapiens
4,PRJDB9974,True,DRR231747,DRX222002,DRS151763,SAMD00229078,DRA010287,RNA-Seq,TRANSCRIPTOMIC,PAIRED,SAMD00229078,Homo sapiens


Let's check what projects we were not able to retrieve using the SRA direct fetching approach

In [49]:
failed_sra_direct = sra_direct[sra_direct.success == False].copy()
failed_sra_direct

,bioproject_accession,success,Run,Experiment,Sample,BioSample,Submission,LibraryStrategy,LibrarySource,LibraryLayout,SampleName,ScientificName
638,-,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23457,PRJEB54580,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95419,PRJEB100787,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95649,PRJEB102956,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
203418,PRJNA658246,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
399638,PRJNA857927,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501865,PRJNA1050789,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
532010,PRJNA1103761,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
552049,PRJNA1136968,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
557418,PRJNA1149772,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Well, for some reason, some of the projects were not retrieved using the SRA direct fetching approach, even though they are present in the SRA database. Take a look here:
- [NCBI SRA search results](https://www.ncbi.nlm.nih.gov/sra/?term=(PRJEB54580%5BGPRJ%5D)+OR+(PRJEB100787%5BGPRJ%5D)+OR+(PRJEB102956%5BGPRJ%5D)+OR+(PRJNA658246%5BGPRJ%5D)+OR+(PRJNA857927%5BGPRJ%5D)+OR+(PRJNA1050789%5BGPRJ%5D)+OR+(PRJNA1103761%5BGPRJ%5D)+OR+(PRJNA1136968%5BGPRJ%5D)+OR+(PRJNA1149772%5BGPRJ%5D)+OR+(PRJNA1190403%5BGPRJ%5D)+OR+(PRJNA1244410%5BGPRJ%5D)+OR+(PRJNA1248477%5BGPRJ%5D)+OR+(PRJNA1371863%5BGPRJ%5D)+OR+(PRJNA1173491%5BGPRJ%5D))
- [NCBI BioProject search results](https://www.ncbi.nlm.nih.gov/bioproject/?term=(PRJEB54580%5BGPRJ%5D)+OR+(PRJEB100787%5BGPRJ%5D)+OR+(PRJEB102956%5BGPRJ%5D)+OR+(PRJNA658246%5BGPRJ%5D)+OR+(PRJNA857927%5BGPRJ%5D)+OR+(PRJNA1050789%5BGPRJ%5D)+OR+(PRJNA1103761%5BGPRJ%5D)+OR+(PRJNA1136968%5BGPRJ%5D)+OR+(PRJNA1149772%5BGPRJ%5D)+OR+(PRJNA1190403%5BGPRJ%5D)+OR+(PRJNA1244410%5BGPRJ%5D)+OR+(PRJNA1248477%5BGPRJ%5D)+OR+(PRJNA1371863%5BGPRJ%5D)+OR+(PRJNA1173491%5BGPRJ%5D))

In [50]:
db = "sra"
query = " OR ".join(
    map(lambda x: f"({x}[GPRJ])", failed_sra_direct.bioproject_accession.tolist()[1:])
)
search_results = eutils_search(
    query=query,
    db=db,
    api_key=NCBI_API_KEY,
)
print(json.dumps(search_results, indent=2))

{
  "header": {
    "type": "esearch",
    "version": "0.3"
  },
  "esearchresult": {
    "count": "0",
    "retmax": "0",
    "retstart": "0",
    "querykey": "1",
    "webenv": "MCID_6a045a086f96b54f2a0c6167",
    "idlist": [],
    "translationset": [],
    "querytranslation": "(PRJEB54580[GPRJ]) OR (PRJEB100787[GPRJ]) OR (PRJEB102956[GPRJ]) OR (PRJNA658246[GPRJ]) OR (PRJNA857927[GPRJ]) OR (PRJNA1050789[GPRJ]) OR (PRJNA1103761[GPRJ]) OR (PRJNA1136968[GPRJ]) OR (PRJNA1149772[GPRJ]) OR (PRJNA1190403[GPRJ]) OR (PRJNA1244410[GPRJ]) OR (PRJNA1248477[GPRJ]) OR (PRJNA1371863[GPRJ]) OR (PRJNA1173491[GPRJ])",
    "errorlist": {
      "phrasesnotfound": [
        "PRJEB54580[GPRJ]",
        "PRJEB100787[GPRJ]",
        "PRJEB102956[GPRJ]",
        "PRJNA658246[GPRJ]",
        "PRJNA857927[GPRJ]",
        "PRJNA1050789[GPRJ]",
        "PRJNA1103761[GPRJ]",
        "PRJNA1136968[GPRJ]",
        "PRJNA1149772[GPRJ]",
        "PRJNA1190403[GPRJ]",
        "PRJNA1244410[GPRJ]",
        "PRJNA124847

In [51]:
db = "bioproject"
query = " OR ".join(
    map(lambda x: f"({x}[GPRJ])", failed_sra_direct.bioproject_accession.tolist()[1:])
)
search_results = eutils_search(
    query=query,
    db=db,
    api_key=NCBI_API_KEY,
)
print(json.dumps(search_results, indent=2))

{
  "header": {
    "type": "esearch",
    "version": "0.3"
  },
  "esearchresult": {
    "count": "7",
    "retmax": "7",
    "retstart": "0",
    "querykey": "1",
    "webenv": "MCID_6a045a0866aa8d2695026e94",
    "idlist": [
      "1380143",
      "1349157",
      "1173491",
      "1149772",
      "1103761",
      "859493",
      "658246"
    ],
    "translationset": [],
    "translationstack": [
      {
        "term": "PRJEB54580[All Fields]",
        "field": "All Fields",
        "count": "1",
        "explode": "N"
      },
      {
        "term": "PRJEB100787[All Fields]",
        "field": "All Fields",
        "count": "1",
        "explode": "N"
      },
      "OR",
      {
        "term": "PRJEB102956[All Fields]",
        "field": "All Fields",
        "count": "1",
        "explode": "N"
      },
      "OR",
      {
        "term": "PRJNA658246[All Fields]",
        "field": "All Fields",
        "count": "1",
        "explode": "N"
      },
      "OR",
      {
        "t

Let's also check whether we retrieved all necessary samples and run and experiment counts are matching for those samples.

In [52]:
grouped_sra_direct = (
    sra_direct.groupby(["bioproject_accession", "Sample"])
    .agg(
        n_experiments=("Experiment", pd.Series.nunique),
        n_runs=("Run", pd.Series.nunique),
        experiments=("Experiment", lambda x: ",".join(x.unique())),
        runs=("Run", lambda x: ",".join(x.unique())),
    )
    .reset_index()
)
grouped_sra_direct.head()

,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
0,PRJDB10650,DRS258235,1,1,DRX239273,DRR249491
1,PRJDB10650,DRS258236,1,1,DRX239274,DRR249492
2,PRJDB10650,DRS258237,1,1,DRX239275,DRR249493
3,PRJDB10917,DRS210942,1,1,DRX248885,DRR259182
4,PRJDB10917,DRS210943,1,1,DRX248886,DRR259183


In [53]:
columns = ["srs", 'prj', "srx", "srr", "n_srx", "n_srr"]
merged_direct_sra_samples = pd.merge(
    samples10x[columns][samples10x.prj.notna()],
    grouped_sra_direct,
    left_on=['prj', 'srs'],
    right_on=['bioproject_accession', 'Sample'],
    how="left"
)
merged_direct_sra_samples.head()

,srs,prj,srx,srr,n_srx,n_srr,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
0,DRS173513,PRJDB9974,DRX222010,DRR231755,1,1,PRJDB9974,DRS173513,1.0,1.0,DRX222010,DRR231755
1,DRS173514,PRJDB9974,DRX222011,DRR231756,1,1,PRJDB9974,DRS173514,1.0,1.0,DRX222011,DRR231756
2,DRS188691,PRJDB11756,"DRX288815,DRX288817","DRR299376,DRR299378",2,2,PRJDB11756,DRS188691,4.0,4.0,"DRX288816,DRX288815,DRX288817,DRX288818","DRR299377,DRR299376,DRR299378,DRR299379"
3,DRS188692,PRJDB11756,"DRX288803,DRX288805","DRR299364,DRR299366",2,2,PRJDB11756,DRS188692,4.0,4.0,"DRX288804,DRX288803,DRX288805,DRX288806","DRR299365,DRR299364,DRR299366,DRR299367"
4,DRS195626,PRJDB8838,DRX185722,DRR195270,1,1,PRJDB8838,DRS195626,1.0,1.0,DRX185722,DRR195270


Okay, let's see if there are any samples outside of the projects that we failed to retrieve using the SRA direct fetching approach that are missing from the retrieved SRA metadata.

In [54]:
failed_sra_direct_samples = merged_direct_sra_samples[
    (merged_direct_sra_samples.isna().any(axis=1))
    & (~merged_direct_sra_samples.prj.isin(
        failed_sra_direct.bioproject_accession.tolist()
    ))
]
failed_sra_direct_samples

,srs,prj,srx,srr,n_srx,n_srr,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
4335,ERS2034568,PRJEB14362,"ERX2958525,ERX2958526,ERX2958527,ERX2958528","ERR2955735,ERR2955736,ERR2955737,ERR2955738",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4336,ERS2034569,PRJEB14362,"ERX2958529,ERX2958530,ERX2958531,ERX2958532","ERR2955739,ERR2955740,ERR2955741,ERR2955742",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4337,ERS2034570,PRJEB14362,"ERX2956238,ERX2956239,ERX2956240,ERX2956241","ERR2953448,ERR2953449,ERR2953450,ERR2953451",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4338,ERS2034571,PRJEB14362,"ERX2958533,ERX2958534,ERX2958535,ERX2958536","ERR2955743,ERR2955744,ERR2955745,ERR2955746",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4386,ERS2063788,PRJEB14362,"ERX2952744,ERX2952745,ERX2952746,ERX2952747","ERR2949954,ERR2949955,ERR2949956,ERR2949957",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4387,ERS2063789,PRJEB14362,"ERX2952748,ERX2952749,ERX2952750,ERX2952751","ERR2949958,ERR2949959,ERR2949960,ERR2949961",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4388,ERS2063790,PRJEB14362,"ERX2952752,ERX2952753,ERX2952754,ERX2952755","ERR2949962,ERR2949963,ERR2949964,ERR2949965",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4389,ERS2063791,PRJEB14362,"ERX2952756,ERX2952757,ERX2952758,ERX2952759","ERR2949966,ERR2949967,ERR2949968,ERR2949969",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4390,ERS2063792,PRJEB14362,"ERX2956033,ERX2956034,ERX2956035,ERX2956036","ERR2953243,ERR2953244,ERR2953245,ERR2953246",4,4,NaN,NaN,NaN,NaN,NaN,NaN
4391,ERS2063793,PRJEB14362,"ERX2956029,ERX2956030,ERX2956031,ERX2956032","ERR2953239,ERR2953240,ERR2953241,ERR2953242",4,4,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
sra_direct[sra_direct.Sample.isin(["SRS15956068", "SRS15956065"])]

,bioproject_accession,success,Run,Experiment,Sample,BioSample,Submission,LibraryStrategy,LibrarySource,LibraryLayout,SampleName,ScientificName
426274,PRJNA826352,True,SRR22518264,SRX18482689,SRS15956065,SAMN31994332,SRA1551324,RNA-Seq,TRANSCRIPTOMIC,PAIRED,D30_ASD1_4,Homo sapiens
426277,PRJNA826352,True,SRR22518261,SRX18482692,SRS15956068,SAMN31994335,SRA1551324,RNA-Seq,TRANSCRIPTOMIC,PAIRED,D30_NC1B_3,Homo sapiens


Okay, it seems like we are missing some of the samples. They all are present in the SRA database, but for some reason they were not retrieved using the SRA direct fetching approach. let's take a look at the esummary results for one of the projects

In [56]:
db = "sra"
search_results = eutils_search(
    query="PRJNA940674",
    db=db,
    api_key=NCBI_API_KEY,
)
# print(json.dumps(search_results, indent=2))

efetch_results = eutils_fetch(
    db="sra",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
    api_key=NCBI_API_KEY,
    rettype="runinfo",
    retmode="text",
)
print(efetch_results)

Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,Experiment,LibraryName,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,InsertSize,InsertDev,Platform,Model,SRAStudy,BioProject,Study_Pubmed_id,ProjectID,Sample,BioSample,SampleType,TaxID,ScientificName,SampleName,g1k_pop_code,source,g1k_analysis_group,Subject_ID,Sex,Disease,Tumor,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
SRR23693884,2023-03-07 21:13:09,2023-03-04 04:47:40,406635252,48389594988,406635252,119,15064,,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos4/sra-pub-run-30/SRR023/23693/SRR23693884/SRR23693884.1,SRX19556102,GSM7078655,RNA-Seq,cDNA,TRANSCRIPTOMIC SINGLE CELL,PAIRED,0,0,ILLUMINA,Illumina NovaSeq 6000,SRP425474,PRJNA940674,,940674,SRS16940755,SAMN33576927,simple,10090,Mus musculus,GSM7078655,,,,,,,no,,,,,YQY711,SRA1598943,,public,2EB445580EEF59FD454CA86DBDB26C99,16A1BC8AB407

Let's try querying by ids

In [58]:
print(json.dumps(search_results, indent=2))

{
  "header": {
    "type": "esearch",
    "version": "0.3"
  },
  "esearchresult": {
    "count": "4",
    "retmax": "4",
    "retstart": "0",
    "querykey": "1",
    "webenv": "MCID_6a045a1b4447a85cf9099e11",
    "idlist": [
      "26844786",
      "26844785",
      "26844784",
      "26844783"
    ],
    "translationset": [],
    "translationstack": [
      {
        "term": "PRJNA940674[All Fields]",
        "field": "All Fields",
        "count": "4",
        "explode": "N"
      },
      "GROUP"
    ],
    "querytranslation": "PRJNA940674[All Fields]"
  }
}


In [60]:
efetch_results = eutils_fetch(
    db="sra",
    ids=search_results["esearchresult"]["idlist"],
    api_key=NCBI_API_KEY,
    rettype="runinfo",
    retmode="text",
)
print(efetch_results)

Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,Experiment,LibraryName,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,InsertSize,InsertDev,Platform,Model,SRAStudy,BioProject,Study_Pubmed_id,ProjectID,Sample,BioSample,SampleType,TaxID,ScientificName,SampleName,g1k_pop_code,source,g1k_analysis_group,Subject_ID,Sex,Disease,Tumor,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
SRR23693884,2023-03-07 21:13:09,2023-03-04 04:47:40,406635252,48389594988,406635252,119,15064,,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos4/sra-pub-run-30/SRR023/23693/SRR23693884/SRR23693884.1,SRX19556102,GSM7078655,RNA-Seq,cDNA,TRANSCRIPTOMIC SINGLE CELL,PAIRED,0,0,ILLUMINA,Illumina NovaSeq 6000,SRP425474,PRJNA940674,,940674,SRS16940755,SAMN33576927,simple,10090,Mus musculus,GSM7078655,,,,,,,no,,,,,YQY711,SRA1598943,,public,2EB445580EEF59FD454CA86DBDB26C99,16A1BC8AB407

Okay, I guess it's the problem with `efetch` not retrieving all the results for some reason, even though the search results indicate that they are present in the database. Let's try using `sra-db-be` api

In [57]:
base = "https://trace.ncbi.nlm.nih.gov/Traces/sra-db-be/sra-"
params = {
    "WebEnv": search_results["esearchresult"]["webenv"],
    "rettype": "runinfo",
    "query_key": search_results["esearchresult"]["querykey"],
}
response = httpx.get(base, params=params)
response.raise_for_status()
print(response.text)

Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,Experiment,LibraryName,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,InsertSize,InsertDev,Platform,Model,SRAStudy,BioProject,Study_Pubmed_id,ProjectID,Sample,BioSample,SampleType,TaxID,ScientificName,SampleName,g1k_pop_code,source,g1k_analysis_group,Subject_ID,Sex,Disease,Tumor,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
SRR23693884,2023-03-07 21:13:09,2023-03-04 04:47:40,406635252,48389594988,406635252,119,15064,,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos4/sra-pub-run-30/SRR023/23693/SRR23693884/SRR23693884.1,SRX19556102,GSM7078655,RNA-Seq,cDNA,TRANSCRIPTOMIC SINGLE CELL,PAIRED,0,0,ILLUMINA,Illumina NovaSeq 6000,SRP425474,PRJNA940674,,940674,SRS16940755,SAMN33576927,simple,10090,Mus musculus,GSM7078655,,,,,,,no,,,,,YQY711,SRA1598943,,public,2EB445580EEF59FD454CA86DBDB26C99,16A1BC8AB407

The result is the same. We are missing some of the samples

Let's check if the samples we managed to retrieve have the correct number of runs and experiments associated with them

In [65]:
# remove samples that are missing from the retrieved SRA metadata
filtered_merged_direct_sra_samples = merged_direct_sra_samples[
    ~(
        (merged_direct_sra_samples.srs.isin(failed_sra_direct_samples.srs.tolist()))
        | (
            merged_direct_sra_samples.prj.isin(
                failed_sra_direct.bioproject_accession.tolist()
            )
        )
    )
]

mismatched_numbers_direct_sra = filtered_merged_direct_sra_samples[
    (
        filtered_merged_direct_sra_samples.n_srx
        != filtered_merged_direct_sra_samples.n_experiments
    )
    | (
        filtered_merged_direct_sra_samples.n_srr
        != filtered_merged_direct_sra_samples.n_runs
    )
]
mismatched_numbers_direct_sra.head()

,srs,prj,srx,srr,n_srx,n_srr,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
2,DRS188691,PRJDB11756,"DRX288815,DRX288817","DRR299376,DRR299378",2,2,PRJDB11756,DRS188691,4.0,4.0,"DRX288816,DRX288815,DRX288817,DRX288818","DRR299377,DRR299376,DRR299378,DRR299379"
3,DRS188692,PRJDB11756,"DRX288803,DRX288805","DRR299364,DRR299366",2,2,PRJDB11756,DRS188692,4.0,4.0,"DRX288804,DRX288803,DRX288805,DRX288806","DRR299365,DRR299364,DRR299366,DRR299367"
167,DRS407504,PRJDB16976,DRX498429,DRR514543,1,1,PRJDB16976,DRS407504,2.0,2.0,"DRX498429,DRX498428","DRR514543,DRR514542"
168,DRS407505,PRJDB16976,DRX498431,DRR514545,1,1,PRJDB16976,DRS407505,2.0,2.0,"DRX498430,DRX498431","DRR514544,DRR514545"
169,DRS407506,PRJDB16976,DRX498433,DRR514547,1,1,PRJDB16976,DRS407506,2.0,2.0,"DRX498433,DRX498432","DRR514547,DRR514546"


It's good, when we retrieve more samples than we expect, I guess. Like in this example:
- [NCBI SRA search results](https://www.ncbi.nlm.nih.gov/sra/?term=DRS188691)

Let's check if there are any samples with more experiments than it should be

In [78]:
mismatched_numbers_direct_sra[
    mismatched_numbers_direct_sra.n_srx > mismatched_numbers_direct_sra.n_experiments
]

,srs,prj,srx,srr,n_srx,n_srr,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
27952,SRS10441854,PRJNA768422,"SRX12478300,SRX18374760","SRR16193799,SRR22405103",2,2,PRJNA768422,SRS10441854,1.0,1.0,SRX12478300,SRR16193799
27953,SRS10441855,PRJNA768422,"SRX12478301,SRX18374761","SRR16193800,SRR22405102",2,2,PRJNA768422,SRS10441855,1.0,1.0,SRX12478301,SRR16193800
27954,SRS10441856,PRJNA768422,"SRX12478302,SRX18374762","SRR16193801,SRR22405101",2,2,PRJNA768422,SRS10441856,1.0,1.0,SRX12478302,SRR16193801
41755,SRS15286089,PRJNA768422,"SRX17758534,SRX18374763","SRR21763376,SRR22405100",2,2,PRJNA768422,SRS15286089,1.0,1.0,SRX17758534,SRR21763376
100352,SRS24456482,PRJNA1240655,"SRX28099581,SRX30601371","SRR32816191,SRR35526028",2,2,PRJNA1240655,SRS24456482,1.0,1.0,SRX28099581,SRR32816191
101786,SRS27109451,PRJNA1337591,"SRX30786436,SRX30786439,SRX30786441,SRX3078646...","SRR35737146,SRR35737149,SRR35737151,SRR3573717...",9,9,PRJNA1337591,SRS27109451,4.0,4.0,"SRX30810104,SRX30810103,SRX30810101,SRX31122606","SRR35760811,SRR35760812,SRR35760814,SRR36076883"
103754,SRS9029085,PRJNA728696,"SRX10950289,SRX10951389","SRR14607158,SRR14608300",2,2,PRJNA728696,SRS9029085,1.0,1.0,SRX10950289,SRR14607158


It seems that samples in study `PRJNA768422` were submitted to SRA twice: once through SRA and once through GEO. Same for `SRS24456482` and `SRS9029085` I guess. You can check it here:
- [NCBI SRA search results for `PRJNA768422`](https://www.ncbi.nlm.nih.gov/sra?linkname=bioproject_sra_all&from_uid=768422)
- [NCBI SRA search results for `SRS24456482`](https://www.ncbi.nlm.nih.gov/sra/?term=SRX28099581+OR+SRX30601371)
- [NCBI SRA search results for `SRS9029085`](https://www.ncbi.nlm.nih.gov/sra/?term=SRX10950289+OR+SRX10951389)


Not so clear for `SRS27109451`. So there is a study [`PRJNA1337591`](https://www.ncbi.nlm.nih.gov/bioproject/?term=PRJNA1337591) with 4 experiments. But there is also study [`PRJNA1345517`](https://www.ncbi.nlm.nih.gov/bioproject/?term=PRJNA1345517) where for the same sample you can find [9 experiments](https://www.ncbi.nlm.nih.gov/sra?term=SAMN52354213). I guess one need to check the publication associated with the project to understand which one is correct

Let's take a loot at the sample with less experiments than expected

In [80]:
mismatched_numbers_direct_sra[
    mismatched_numbers_direct_sra.n_srx < mismatched_numbers_direct_sra.n_experiments
]

,srs,prj,srx,srr,n_srx,n_srr,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
2,DRS188691,PRJDB11756,"DRX288815,DRX288817","DRR299376,DRR299378",2,2,PRJDB11756,DRS188691,4.0,4.0,"DRX288816,DRX288815,DRX288817,DRX288818","DRR299377,DRR299376,DRR299378,DRR299379"
3,DRS188692,PRJDB11756,"DRX288803,DRX288805","DRR299364,DRR299366",2,2,PRJDB11756,DRS188692,4.0,4.0,"DRX288804,DRX288803,DRX288805,DRX288806","DRR299365,DRR299364,DRR299366,DRR299367"
167,DRS407504,PRJDB16976,DRX498429,DRR514543,1,1,PRJDB16976,DRS407504,2.0,2.0,"DRX498429,DRX498428","DRR514543,DRR514542"
168,DRS407505,PRJDB16976,DRX498431,DRR514545,1,1,PRJDB16976,DRS407505,2.0,2.0,"DRX498430,DRX498431","DRR514544,DRR514545"
169,DRS407506,PRJDB16976,DRX498433,DRR514547,1,1,PRJDB16976,DRS407506,2.0,2.0,"DRX498433,DRX498432","DRR514547,DRR514546"
...,...,...,...,...,...,...,...,...,...,...,...,...
103762,SRS9109834,PRJNA734283,"SRX11040903,SRX11040902,SRX11040901","SRR14702873,SRR14702874,SRR14702875",3,3,PRJNA734283,SRS9109834,8.0,8.0,"SRX11040899,SRX11040898,SRX11040897,SRX1104089...","SRR14702877,SRR14702878,SRR14702879,SRR1470288..."
103781,SRS9161836,PRJNA736082,SRX11096745,SRR14763274,1,1,PRJNA736082,SRS9161836,2.0,2.0,"SRX11096748,SRX11096745","SRR14763271,SRR14763274"
103782,SRS9161837,PRJNA736082,SRX11096746,SRR14763273,1,1,PRJNA736082,SRS9161837,2.0,2.0,"SRX11096749,SRX11096746","SRR14763270,SRR14763273"
103783,SRS9161838,PRJNA736082,SRX11096744,SRR14763275,1,1,PRJNA736082,SRS9161838,2.0,2.0,"SRX11096747,SRX11096744","SRR14763272,SRR14763275"


In the case of sample [`DRS188691`](https://www.ncbi.nlm.nih.gov/sra/?term=DRS188691) we can see that reads for one sequencing run were submitted to separate SRA runs and for each SRA experiment there are two such run pairs. Total mess.

In the case of sample [`SRS9161836`](https://www.ncbi.nlm.nih.gov/sra/?term=SRS9161836) you can see that CITE-seq modality was submitted along side the scRNA-seq modality for each experiment, so there are [2 runs per experiment](https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SAMN19608335&o=acc_s%3Aa)

In [83]:
mismatched_numbers_direct_sra[
    mismatched_numbers_direct_sra.n_srx < mismatched_numbers_direct_sra.n_experiments
].to_csv("data/mismatched_experiments_direct_sra.csv", index=False)

Finally, let's see if there are any samples with mathcing number of experiments but mismatching number of runs

In [82]:
mismatched_numbers_direct_sra[
    (mismatched_numbers_direct_sra.n_srx == mismatched_numbers_direct_sra.n_experiments)
    & (mismatched_numbers_direct_sra.n_srr != mismatched_numbers_direct_sra.n_runs)
]

,srs,prj,srx,srr,n_srx,n_srr,bioproject_accession,Sample,n_experiments,n_runs,experiments,runs
334,DRS524546,PRJDB19651,DRX730719,DRR751152,1,1,PRJDB19651,DRS524546,1.0,2.0,DRX730719,"DRR751152,DRR751153"
335,DRS524547,PRJDB19651,DRX730720,"DRR751155,DRR751157",1,2,PRJDB19651,DRS524547,1.0,4.0,DRX730720,"DRR751157,DRR751156,DRR751154,DRR751155"
336,DRS524548,PRJDB19651,DRX730721,"DRR751159,DRR751161",1,2,PRJDB19651,DRS524548,1.0,4.0,DRX730721,"DRR751158,DRR751160,DRR751161,DRR751159"
424,DRS556163,PRJDB32878,DRX766409,DRR787602,1,1,PRJDB32878,DRS556163,1.0,2.0,DRX766409,"DRR787602,DRR787601"
425,DRS556164,PRJDB32878,DRX766410,DRR787604,1,1,PRJDB32878,DRS556164,1.0,2.0,DRX766410,"DRR787604,DRR787603"
...,...,...,...,...,...,...,...,...,...,...,...,...
93784,SRS27095642,PRJNA1138229,SRX31108345,SRR36061161,1,1,PRJNA1138229,SRS27095642,1.0,2.0,SRX31108345,"SRR36061160,SRR36061161"
94896,SRS12996784,PRJNA837932,SRX15265850,"SRR19201063,SRR19222029,SRR19222030,SRR1922203...",1,8,PRJNA837932,SRS12996784,1.0,9.0,SRX15265850,"SRR19201063,SRR19222029,SRR19222030,SRR1922203..."
94897,SRS12996786,PRJNA837932,SRX15265852,"SRR19201061,SRR19221938,SRR19221939,SRR1922194...",1,8,PRJNA837932,SRS12996786,1.0,9.0,SRX15265852,"SRR19201061,SRR19221938,SRR19221939,SRR1922194..."
102102,SRS4415503,PRJNA524398,"SRX5437900,SRX5437877,SRX5437878,SRX5437879,SR...","SRR8831269,SRR8832283,SRR8832284,SRR8832435,SR...",12,23,PRJNA524398,SRS4415503,12.0,24.0,"SRX5437900,SRX5437877,SRX5437878,SRX5437879,SR...","SRR8831268,SRR8831269,SRR8832283,SRR8832284,SR..."


Okay, it seems like it's also a list of problematic submissions. For example experiment [DRX730719](https://www.ncbi.nlm.nih.gov/sra/?term=DRX730719) has 2 runs associated with it. And if you go to the run viewer you can see that one of the runs has average leangth of 20. Not exactly sure what it is. Potentially they submitted indexes separately from the reads

Let's save the list of samples with mismatching number of runs for futher investigation

In [84]:
mismatched_numbers_direct_sra[
    (mismatched_numbers_direct_sra.n_srx == mismatched_numbers_direct_sra.n_experiments)
    & (mismatched_numbers_direct_sra.n_srr != mismatched_numbers_direct_sra.n_runs)
].to_csv("data/mismatched_runs_direct_sra.csv", index=False)